In [1]:
#install libraries
!pip -q install pandas numpy requests beautifulsoup4 lxml tqdm sentence-transformers faiss-cpu transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.8 MB/s eta 0:00:00


In [2]:
#imports + constants
import re, time, random, json
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://givana.my"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; GivanaRAGBot/1.0)"}

# WooCommerce product category sitemap you used earlier
PRODUCT_CAT_SITEMAP = BASE_URL.rstrip("/") + "/taxonomy-type-product_cat-sitemap-1.xml"

In [3]:
#Helper: Read URLs from sitemap
def fetch_urls_from_sitemap(sitemap_url, limit=500):
    r = requests.get(sitemap_url, timeout=25, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "xml")
    return [loc.get_text().strip() for loc in soup.find_all("loc")][:limit]

In [4]:
#Helper: Extract product links from the category listing pages
def extract_product_links_from_listing(listing_url):
    r = requests.get(listing_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return set()
    soup = BeautifulSoup(r.text, "lxml")

    links = set()
    for a in soup.select("a[href*='/product/']"):
        href = a.get("href")
        if not href:
            continue
        full = urljoin(listing_url, href)
        full = full.split("?")[0].split("#")[0].rstrip("/")
        if full.startswith(BASE_URL) and "/product/" in full:
            links.add(full)
    return links

In [5]:
#Helper: Get product name from product page (H1)
def get_product_name(product_url):
    r = requests.get(product_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    # fallback
    title = soup.title.get_text(strip=True) if soup.title else ""
    if title:
        return title.split(" - ")[0].strip()
    return None

In [7]:
def crawl_shop_products(base_url, max_pages=5):
    product_links = set()

    for page in range(1, max_pages + 1):
        if page == 1:
            url = f"{base_url}/shop/"
        else:
            url = f"{base_url}/shop/page/{page}/"

        try:
            r = requests.get(url, timeout=20, headers=HEADERS)
            if r.status_code != 200:
                break

            soup = BeautifulSoup(r.text, "lxml")

            links_on_page = 0
            for a in soup.select("a[href*='/product/']"):
                href = a.get("href")
                if href:
                    full = urljoin(url, href)
                    full = full.split("?")[0].split("#")[0].rstrip("/")
                    if full.startswith(BASE_URL):
                        product_links.add(full)
                        links_on_page += 1

            # if no products found on page, stop pagination
            if links_on_page == 0:
                break

            time.sleep(0.5)

        except Exception as e:
            print("Shop crawl error:", e)
            break

    return sorted(product_links)

In [8]:
current_product_urls = crawl_shop_products(BASE_URL, max_pages=5)

print("Products found from shop:", len(current_product_urls))
print(current_product_urls[:15])

Shop crawl error: HTTPSConnectionPool(host='givana.my', port=443): Read timed out. (read timeout=20)
Products found from shop: 9
['https://givana.my/product/box', 'https://givana.my/product/cookies-with-love', 'https://givana.my/product/for-ameer', 'https://givana.my/product/raya-for-her', 'https://givana.my/product/sakinah-gift', 'https://givana.my/product/salam-kareem', 'https://givana.my/product/the-beary-crush', 'https://givana.my/product/the-tranquil-solat', 'https://givana.my/product/wheres-my-love']


**The above two cells are extracting the products from the shop with pagination and run shop crawler. In that case, it scrapes the products where the customers actually see and it has less server load and less timeout risk.**

In [11]:
#Build products_df from the 9 urls
#Extract product names (H1) and create products_df
def get_product_name(product_url):
    r = requests.get(product_url, timeout=25, headers=HEADERS)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "lxml")
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)
    title = soup.title.get_text(strip=True) if soup.title else ""
    return title.split(" - ")[0].strip() if title else None

products = []
for u in tqdm(current_product_urls):
    try:
        name = get_product_name(u)
        if name:
            products.append({"name": name, "url": u})
        time.sleep(0.2)
    except Exception as e:
        print("Name fail:", u, e)

products_df = pd.DataFrame(products).drop_duplicates().sort_values("name").reset_index(drop=True)
print("Current products found:", len(products_df))
products_df

100%|██████████| 9/9 [00:27<00:00,  3.06s/it]

Current products found: 9


,name,url
0,Cookies with Love,https://givana.my/product/cookies-with-love
1,Emerald Indulgence,https://givana.my/product/box
2,For Ameer,https://givana.my/product/for-ameer
3,Raya For Her,https://givana.my/product/raya-for-her
4,Sakinah Gift,https://givana.my/product/sakinah-gift
5,Salam Kareem,https://givana.my/product/salam-kareem
6,The Beary Crush,https://givana.my/product/the-beary-crush
7,The Tranquil Solat,https://givana.my/product/the-tranquil-solat
8,Where’s My Love,https://givana.my/product/wheres-my-love


In [12]:
#Crawl only 9 products
#Scraper
def clean_text(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip()

def scrape_page(url: str) -> dict:
    r = requests.get(url, timeout=25, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    for tag in soup(["script","style","noscript","header","footer","nav"]):
        tag.decompose()

    title = soup.title.get_text(strip=True) if soup.title else url
    text = clean_text(soup.get_text(" "))
    return {"url": url, "title": title, "text": text[:15000]}

In [13]:
#Build docs_raw
docs_raw = []
for u in tqdm(products_df["url"].tolist()):
    try:
        docs_raw.append(scrape_page(u))
        time.sleep(0.25)
    except Exception as e:
        print("Scrape fail:", u, e)

print("docs_raw pages:", len(docs_raw))
print(docs_raw[0]["title"] if docs_raw else "No docs scraped")

100%|██████████| 9/9 [00:29<00:00,  3.23s/it]

docs_raw pages: 9
Cookies with Love -


In [14]:
#Chunking
def chunk_text(text: str, chunk_size=220, overlap=40):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+chunk_size]))
        i += max(1, chunk_size - overlap)
    return chunks

chunks = []
for d in docs_raw:
    for idx, ch in enumerate(chunk_text(d["text"], chunk_size=220, overlap=40)):
        chunks.append({
            "chunk_id": f'{d["url"]}__{idx}',
            "url": d["url"],
            "title": d["title"],
            "text": ch
        })

chunks_df = pd.DataFrame(chunks)
print("chunks_df:", chunks_df.shape)
chunks_df.head(3)

chunks_df: (18, 4)


,chunk_id,url,title,text
0,https://givana.my/product/cookies-with-love__0,https://givana.my/product/cookies-with-love,Cookies with Love -,Cookies with Love - Skip to content Sale! Cook...
1,https://givana.my/product/cookies-with-love__1,https://givana.my/product/cookies-with-love,Cookies with Love -,The Tranquil Solat RM 49.00 Add to cart Scroll...
2,https://givana.my/product/box__0,https://givana.my/product/box,Emerald Indulgence -,Emerald Indulgence - Skip to content Emerald I...


In [15]:
#Embedding and FAISS
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = chunks_df["text"].tolist()
emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
emb = np.array(emb).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

print("FAISS index size:", index.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index size: 18


In [16]:
#Retriever and clean retriever
def retrieve(query: str, k=5):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    out = []
    for score, i in zip(scores[0], ids[0]):
        row = chunks_df.iloc[int(i)]
        out.append({
            "score": float(score),
            "url": row["url"],
            "title": row["title"],
            "text": row["text"]
        })
    return out

NOISE = [
    "add to cart", "reviews", "be the first to review", "your rating",
    "cancel reply", "email address will not be published", "sku", "quantity",
]

def clean_context(text: str) -> str:
    tl = text.lower()
    for p in NOISE:
        tl = tl.replace(p, " ")
    tl = re.sub(r"\s+", " ", tl).strip()
    return tl

def retrieve_clean(query: str, k=3):
    res = retrieve(query, k=k)
    cleaned = []
    for r in res:
        ct = clean_context(r["text"])
        if len(ct) > 120:
            cleaned.append({**r, "clean_text": ct})
    return cleaned[:k]

In [17]:
#Test retrieval
test_name = products_df["name"].iloc[0]
hits = retrieve_clean(test_name, k=3)
print("Test product:", test_name)
for h in hits:
    print("\n---", round(h["score"], 3), h["url"])
    print(h["clean_text"][:220], "...")

Test product: Cookies with Love

--- 0.611 https://givana.my/product/cookies-with-love
cookies with love - skip to content sale! cookies with love rm 95.00 original price was: rm95.00. rm 90.00 current price is: rm90.00. complete your gift with a card & message (optional) − cookies with love + description  ...

--- 0.429 https://givana.my/product/wheres-my-love
where’s my love - skip to content where’s my love rm 60.00 complete your gift with a card & message (optional) − where's my love + description additional information (0) this box contains: love perfume earring and broch  ...

--- 0.377 https://givana.my/product/the-beary-crush
the beary crush - skip to content the beary crush rm 87.00 complete your gift with a card & message (optional) − the beary crush + description additional information (0) this box contains: aura perfume teddy candle capyb ...


In [32]:
#Load the LLM
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_NAME = "google/flan-t5-base"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME)

llm_device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(llm_device)

print("Loaded:", LLM_NAME, "| device:", llm_device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded: google/flan-t5-base | device: cpu


In [33]:
#Stable generation outputs (No weird outputs)
def llm_generate(prompt: str, max_new_tokens=170):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(llm_device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5,
        repetition_penalty=1.2,
        early_stopping=True
    )
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [34]:
#RAG Caption using retrieved context
def rag_caption_llm(product_name, post_type, theme, k=3):
    ctx = retrieve_clean(f"{product_name} {theme} {post_type}", k=k)
    context = "\n".join([f"- {c['clean_text'][:240]}" for c in ctx])[:1100]

    prompt = f"""
You write social media captions for Givana, a Malaysian online gift shop.

Rules:
- English only
- Warm, premium, minimal
- No unrelated stories, no country names, no random names
- 2 short paragraphs maximum
- 1 emoji maximum
- Mention the product name exactly once
- End with ONE soft CTA: "Order on our website" OR "WhatsApp us for customisation"
- Create 8 to 12 hashtags (Malaysia + gifting + occasion). Avoid repeating the exact same hashtag set every time.

Post type: {post_type}
Theme/occasion: {theme}
Product: {product_name}

Use only facts from this context:
{context}

Return exactly in this format:
Caption: ...
Hashtags: ...
"""
    out = llm_generate(prompt)

    caption, hashtags = out, ""
    if "Hashtags:" in out:
        a, b = out.split("Hashtags:", 1)
        caption = a.replace("Caption:", "").strip()
        hashtags = b.strip()
    else:
        hashtags = "#Givana #GivanaMY #GiftMalaysia #GiftIdeas #GiftingMadeEasy #Malaysia #GiftBox #Gifting"

    hashtags = re.sub(r"\s+", " ", hashtags).strip()
    return caption, hashtags, ctx

In [35]:
#Quick test to make sure if it writes normally
POST_TYPES = ["Product Spotlight","Gifting Tip","FAQ","Relatable Funny","Mini Story","Occasion Reminder"]
THEMES = ["Hari Raya","Birthday","Thank You","Anniversary","Corporate","Just Because"]

p = products_df["name"].iloc[0]
caption, hashtags, ctx = rag_caption_llm(p, "Product Spotlight", "Birthday", k=3)
print(caption)
print("\n", hashtags)

Givana - Malaysian online gift shop

 #Givana #GivanaMY #GiftMalaysia #GiftIdeas #GiftingMadeEasy #Malaysia #GiftBox #Gifting


In [36]:
#Similarity memory
history_embs = []

def max_similarity_to_history(caption: str) -> float:
    if not history_embs:
        return 0.0
    e = embedder.encode([caption], normalize_embeddings=True).astype("float32")[0]
    sims = np.dot(np.vstack(history_embs), e)
    return float(np.max(sims))

def remember_caption(caption: str):
    history_embs.append(embedder.encode([caption], normalize_embeddings=True).astype("float32")[0])

In [37]:
#Simple time suggestion
def suggest_time(day_idx):
    dow = day_idx % 7
    if dow in [5, 6]:  # weekend
        return random.choice(["11:00", "21:00"])
    return random.choice(["12:30", "20:45"])

In [38]:
#Generate one unique post with retries
def generate_unique_post(day_idx, product_name, threshold=0.78, tries=20):
    post_type = POST_TYPES[day_idx % len(POST_TYPES)]
    theme = random.choice(THEMES)

    best = None
    best_sim = 1.0

    for _ in range(tries):
        caption, hashtags, ctx = rag_caption_llm(product_name, post_type, theme, k=3)
        sim = max_similarity_to_history(caption)

        if sim < threshold and len(caption) > 50:
            remember_caption(caption)
            return {
                "day": day_idx + 1,
                "product": product_name,
                "post_type": post_type,
                "theme": theme,
                "suggested_time": suggest_time(day_idx),
                "caption": caption,
                "hashtags": hashtags,
                "max_similarity": round(sim, 3),
                "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
            }

        if sim < best_sim:
            best_sim = sim
            best = (caption, hashtags, ctx, post_type, theme)

        post_type = random.choice(POST_TYPES)
        theme = random.choice(THEMES)

    caption, hashtags, ctx, post_type, theme = best
    remember_caption(caption)
    return {
        "day": day_idx + 1,
        "product": product_name,
        "post_type": post_type,
        "theme": theme,
        "suggested_time": suggest_time(day_idx),
        "caption": caption,
        "hashtags": hashtags,
        "max_similarity": round(best_sim, 3),
        "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
    }

In [39]:
# Cache context per product so we don't retrieve every time
product_context_cache = {}

def get_cached_context(product_name, k=2):
    if product_name not in product_context_cache:
        ctx = retrieve_clean(product_name, k=k)  # simple query works well
        context = "\n".join([f"- {c['clean_text'][:240]}" for c in ctx])[:800]
        product_context_cache[product_name] = (ctx, context)
    return product_context_cache[product_name]

In [40]:
#faster version of LLM
def rag_caption_llm(product_name, post_type, theme, k=2):
    ctx, context = get_cached_context(product_name, k=k)

    prompt = f"""
You write social media captions for Givana, a Malaysian online gift shop.

Rules:
- English only
- Warm, premium, minimal
- No unrelated stories, no random names or countries
- 2 short paragraphs max
- 1 emoji max
- Mention the product name exactly once
- End with ONE soft CTA: "Order on our website" OR "WhatsApp us for customisation"
- Create 8 to 12 hashtags (vary them)

Post type: {post_type}
Theme/occasion: {theme}
Product: {product_name}

Use only facts from this context:
{context}

Return exactly:
Caption: ...
Hashtags: ...
"""
    out = llm_generate(prompt, max_new_tokens=120)

    caption, hashtags = out, ""
    if "Hashtags:" in out:
        a, b = out.split("Hashtags:", 1)
        caption = a.replace("Caption:", "").strip()
        hashtags = b.strip()
    else:
        hashtags = "#Givana #GivanaMY #GiftMalaysia #GiftIdeas #GiftingMadeEasy #Malaysia #GiftBox #Gifting"

    hashtags = re.sub(r"\s+", " ", hashtags).strip()
    return caption, hashtags, ctx

In [42]:
#Make generation LLM faster
def llm_generate(prompt: str, max_new_tokens=120):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(llm_device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=2,              # ✅ was 5 (slower)
        repetition_penalty=1.15,
        early_stopping=True
    )
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [44]:
def generate_unique_post(day_idx, product_name, threshold=0.80):
    post_type = POST_TYPES[day_idx % len(POST_TYPES)]
    theme = random.choice(THEMES)

    caption, hashtags, ctx = rag_caption_llm(product_name, post_type, theme, k=2)

    sim = max_similarity_to_history(caption)

    remember_caption(caption)

    return {
        "day": day_idx + 1,
        "product": product_name,
        "post_type": post_type,
        "theme": theme,
        "suggested_time": suggest_time(day_idx),
        "caption": caption,
        "hashtags": hashtags,
        "max_similarity": round(sim, 3),
        "sources": "; ".join(sorted(set([c["url"] for c in ctx])))
    }

In [45]:
history_embs.clear()

products_list = products_df["name"].tolist()

calendar = []
for i in range(30):
    product_name = products_list[i % len(products_list)]
    calendar.append(generate_unique_post(i, product_name, threshold=0.80))

calendar_df = pd.DataFrame(calendar)
calendar_df.head(10)

,day,product,post_type,theme,suggested_time,caption,hashtags,max_similarity,sources
0,1,Cookies with Love,Product Spotlight,Corporate,12:30,"Givana, ...",#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,0.000,https://givana.my/product/cookies-with-love; h...
1,2,Emerald Indulgence,Gifting Tip,Corporate,12:30,Givana is promoting its online gift shop.,,0.553,https://givana.my/product/box; https://givana....
2,3,For Ameer,FAQ,Anniversary,20:45,Givana - Malaysian online gift shop,#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,0.820,https://givana.my/product/for-ameer; https://g...
3,4,Raya For Her,Relatable Funny,Birthday,20:45,"Givana, emerald indulgence",#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,0.641,https://givana.my/product/box; https://givana....
4,5,Sakinah Gift,Mini Story,Hari Raya,20:45,"Givana, emerald indulgence",#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,1.000,https://givana.my/product/box; https://givana....
5,6,Salam Kareem,Occasion Reminder,Corporate,21:00,Givana - Malaysian online gift shop,#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,1.000,https://givana.my/product/for-ameer; https://g...
6,7,The Beary Crush,Product Spotlight,Thank You,11:00,Givana is promoting the Beary Crush.,#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,0.617,https://givana.my/product/salam-kareem; https:...
7,8,The Tranquil Solat,Gifting Tip,Hari Raya,12:30,Givana - Malaysian online gift shop,#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,1.000,https://givana.my/product/raya-for-her; https:...
8,9,Where’s My Love,FAQ,Thank You,12:30,"Givana, ...",#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,1.000,https://givana.my/product/cookies-with-love; h...
9,10,Cookies with Love,Relatable Funny,Thank You,20:45,"Givana, 'Giviana is the best gift shop in Mala...",#Givana #GivanaMY #GiftMalaysia #GiftIdeas #Gi...,0.843,https://givana.my/product/cookies-with-love; h...


In [46]:
#Check similarity stats (quick quality check)
calendar_df["max_similarity"].describe()


,max_similarity
count,30.000000
mean,0.897467
std,0.212958
min,0.000000
25%,0.865250
50%,1.000000
75%,1.000000
max,1.000000


In [48]:
#Export csv and download
calendar_df.to_csv("givana_30_day_calendar_LLM_RAG.csv", index=False)

from google.colab import files
files.download("givana_30_day_calendar_LLM_RAG.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>